# Batch Normalization

## Introduction
Machine Learning and Artificial Intelligence have been able to solve complex problems in the field of vision, speech, text and many other areas. This has been done with the help of training models made up of learnable weights and biases where these weights and biases learn information stored in data through the help of gradient propagation. In models such as logistic regression, xgboost we have a single layer that trains on the input data. Here the learnable weights and biases are only dependent on the input data. But when it comes to **deep neural networks** we have layers built over layers with multiple trainable weights and biases. The input to each layer depends upon the parameters of all of the preceeding layers. Small changes to the network parameters amplify as the network becomes deeper. 

## Effect of Small Changes Made to Trainable Parameters
To see what it means when we say the layers of a neural network gets affected by small changes in the earlier layer, we will have to look at an example. We have an input tensor of shape (1, 27) containing random values initialized from a normal distribution and a neural network model that has 8 layers each layer returns 27 values as their output.

In [1]:
import torch
torch.manual_seed(42)

# Initializing input from a normal distribution
input_tensor = torch.randn(size=(1, 27))

input_tensor

tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047,
         -0.7521,  1.6487, -0.3925,  0.2415, -1.1109,  0.0915, -2.3169, -0.2168,
         -1.3847, -0.8712, -0.2234, -0.6216, -0.5920, -0.0631, -0.8286,  0.3309,
         -1.5576,  0.9956, -0.8798]])

Initializing 8 layers through which the `input_tensor` will propagate through.

In [2]:
torch.manual_seed(42)

layer_1 = torch.nn.Linear(in_features=27, out_features=27, bias=True)
layer_2 = torch.nn.Linear(in_features=27, out_features=27, bias=True)
layer_3 = torch.nn.Linear(in_features=27, out_features=27, bias=True)
layer_4 = torch.nn.Linear(in_features=27, out_features=27, bias=True)
layer_5 = torch.nn.Linear(in_features=27, out_features=27, bias=True)
layer_6 = torch.nn.Linear(in_features=27, out_features=27, bias=True)
layer_7 = torch.nn.Linear(in_features=27, out_features=27, bias=True)
layer_8 = torch.nn.Linear(in_features=27, out_features=27, bias=True)

Below is the result of propagation of `input_tensor` through the 8 neural network layers

In [3]:
with torch.inference_mode():
    output = layer_8(layer_7(layer_6(layer_5(layer_4(layer_3(layer_2(layer_1(input_tensor))))))))
output

tensor([[-0.0962,  0.0791, -0.2677,  0.0717, -0.2088, -0.0038, -0.0842, -0.2494,
         -0.0754, -0.0752, -0.0751,  0.0547,  0.0109, -0.1244, -0.1064,  0.2119,
         -0.0659,  0.0616, -0.1572, -0.1072,  0.0664,  0.0715,  0.1181,  0.0165,
         -0.0165, -0.0356, -0.1565]])

Now we will see the impact of what happens to the output of the 8th layer when we add a small change to weights of the first two layers, the first 4 layers and the first 7 layers. This change will be same for each layer.

In [4]:
torch.manual_seed(42)

layer_1.weight = torch.nn.Parameter(layer_1.weight + 0.1)
layer_2.weight = torch.nn.Parameter(layer_2.weight + 0.1)

with torch.inference_mode():
    output = layer_8(layer_7(layer_6(layer_5(layer_4(layer_3(layer_2(layer_1(input_tensor))))))))

print("Output of the 8th layer after adding a small change to only the first two layers")
print(output)

layer_3.weight = torch.nn.Parameter(layer_3.weight + 0.1)
layer_4.weight = torch.nn.Parameter(layer_4.weight + 0.1)

with torch.inference_mode():
    output = layer_8(layer_7(layer_6(layer_5(layer_4(layer_3(layer_2(layer_1(input_tensor))))))))

print("Output of the 8th layer after adding a small change to only the first four layers")
print(output)

layer_5.weight = torch.nn.Parameter(layer_5.weight + 0.1)
layer_6.weight = torch.nn.Parameter(layer_6.weight + 0.1)
layer_7.weight = torch.nn.Parameter(layer_7.weight + 0.1)

with torch.inference_mode():
    output = layer_8(layer_7(layer_6(layer_5(layer_4(layer_3(layer_2(layer_1(input_tensor))))))))

print("Output of the 8th layer after adding a small change to first seven layers")
print(output)

Output of the 8th layer after adding a small change to only the first two layers
tensor([[-0.0151,  0.0990, -0.4402,  0.0838, -0.1651,  0.0786, -0.0406, -0.2421,
         -0.1082, -0.1180, -0.0980, -0.0005,  0.0197,  0.0017, -0.1518,  0.2673,
         -0.0930, -0.0522, -0.1637,  0.0114, -0.0527, -0.0425,  0.1619,  0.1025,
         -0.0098, -0.0888, -0.2100]])
Output of the 8th layer after adding a small change to only the first four layers
tensor([[-0.7808, -2.3111,  2.5416,  1.4882, -0.2335, -0.6388,  1.3370,  4.5843,
          0.6710, -0.0327,  4.3541, -1.1272,  0.1528, -0.9248, -1.9997,  1.0341,
         -1.6927, -0.1582,  3.3578, -5.3192,  2.0461,  4.3197, -2.5511,  0.7728,
          0.2968, -2.0484,  2.3346]])
Output of the 8th layer after adding a small change to first seven layers
tensor([[ -24.0183,   79.9379,  -55.5321, -278.0000,  338.5512,   66.4865,
           94.0162,   38.9852, -248.3757, -358.0836, -316.8322, -231.1112,
         -100.4568, -137.6178,  188.9039, -352.7201

We can see that there is a large impact on the inputs of deeper layers even with a small change in the neural network weights and biases. In this example we only had 1 input tensor but when it comes to deep neural networks we are training on multiple data points which means that every time a new data point is provided the distribution of the output layers change. 

The change in the distributions of layers’ inputs presents a problem because the layers need to continuously adapt to the new distribution. Having a fixed distribution has an advantage of making training a neural network more efficient. 

The paper [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/pdf/1502.03167) provides a solution to eliminate this inefficient process of training. The change in the distribution of network activations due to the change in network parameters during training is what the authors of the paper call **Internal Covariate Shift**. The authors seek to reduce this internal covariate shift during the training process.

It has been generally observed that the network training converges faster as we whiten the image i.e. we shift the input to have a mean 0 and variance of 1. The authors propose to have this transformation take place for each layer instead of whitening the input to the model.

## Normalization via Batch Statistics
In Deep Learning training of models takes place over a batch of input at a time. So at any given time during the training phase we only have access to the batch statistics. In batch normalization a set of **runnning statistics** is kept to which some part of each batch statistics is added so that after the entire training is over we have statistics to normalize input during the inference stage which has been influenced by data present across all batches.

So the initial goal of batch normalization is to whiten the output of the current layer by scaling and shifting the output to have 0 mean and unit variance. But this would mean we remove the representational power each layer holds as we go through the normalization process. The layers should have the ability to restore their **Representational Power**. They give network the capacity to undo the **normalization**, not that the network will necessarily choose to do so.

Forcing a layer's activations to have a mean of $0$ and a variance of $1$ can restrict the network. For example, if the subsequent layer is a Sigmoid activation function, constraining inputs strictly between $-1$ and $1$ forces the network to only use the linear and middle section of the Sigmoid curve to update the weights wasting the non-linear properties at the ends (to understand this have a look at `pytorch-fundamentals/notebooks/activation_functions.ipynb`)

By introducing the learnable parameters $\gamma$ and $\beta$, gradient descent is permitted to shift and scale the normalized data to wherever it minimizes the loss function. If normalizing the data to a mean of $0$ and variance of $1$ hurts the network's performance backpropagation will adjust $\gamma$ and $\beta$ toward the original standard deviation and mean to approximate $y = x$ .If the network needs a mean of $5$ and a standard deviation of $0.2$ to optimally activate the next layer backpropagation will update $\gamma \approx 0.2$ and $\beta \approx 5$. Ultimately, $\gamma$ and $\beta$ decouple the scale and shift of the activations from the statistics of the raw data. They will only equal the running mean and variance if returning the exact unnormalized input yields the lowest possible loss.

Every **Batch Normalization** layer therefore has been given the ability to restore the representational capability of the network layer if it leads to better learning of the features and better loss minimalization using two trainable weights $\gamma$ and $\beta$

## Batch Normalization Process

There is a dedicated layer that perform normalization called **Batch Normalization** layer. The process of normalization takes place during the forward propagation of the input. For every batch normalization layer we need to initialize 6 different components

1. $\lambda$: This is the trainable **weight** that scales the normalized input to restore the representation power of the incoming layer. This is initialized to a value of 0.
2. $\beta$: Theis the trainable **bias** that shifts the normalized input to restore the representation power of the incoming layer. This is initialized to a value of 1.
3. $\epsilon$: This represents epsilon to prevent computational overflow.
4. **Running Mean**: This will represent the **unbiased** mean of the entire dataset at the end of the training process. This is initialized to a value of 0.
5. **Running Variance**: This will represent **unbiased** variance of the entire dataset at the end of the training process. This is initialized to a value of 1.
6. **momentum**: This is the factor used to update running mean and variance through the process of exponential moving average using batch mean and variance.

### Process of Updating Population Statistics
During the inference mode when the training of the model is complete we might not event get batched inputs for the model to perform inference on, Here we would there need some form of mean and variance values that will allow us to perform normalization step. To do so we have a running mean and variance calculated using **Exponential Moving Average**. The **Exponential Moving Average** is the process of keeping a record of a statistic as part of previous value and part of the incoming value. **momentum** decides how much of the new incoming value will be responsible for future process and (1 - **momentum**) determines how much of the previous value will be responsible for the future process. The equation is: <br>
$$MovingAverage_{n} = ((1 - mommentum) * MovingAverage_{n - 1}) + (momentum * IncomingValue)$$

We use Exponential Moving Average to calculate the running mean and variance for the dataset. The values though intialized as 0 and 1 respectively will converge to the true mean and unbiased variance of the entire dataset.

In general in most of our cases we are dealing with a 2D input (a batch of linear features) **(N, L)** and 4D input (a batch of images) **(N, C, H, W)**. We need to decide how many trainable weights and biases are needed along with decide how many sets of running mean and variance values are to be used. This depends upon the number of **dimensions** across which we are tring to perform the normalization process.

#### For 2D dataset
2D dataset represent data which contain linear features of shape (N, L) where N is the size of the batch and L is the number of features associated with a single data point. In this case the number of unique dimensions is L. So we will have 
- $\lambda$ of shape $(1, L)$
- $\beta$ of shape $(1, L)$
- **Running Mean** of shape $(1, L)$
- **Running Variance** of shape $(1, L)$

#### For 4D datasets
For 4D datasets such as those where we have images as inputs of shape (N, C, H, W) here C is responsible for a different dimension. In this case the number of unique dimensions is C. So we will have
- $\lambda$ of shape $(1, C, 1, 1)$
- $\beta$ of shape $(1, C, 1, 1 )$
- **Running Mean** of shape $(1, C, 1, 1)$
- **Running Variance** of shape $(1, C, 1, 1)$

In case of 4D datasets normalization takes place across the axes (0, 2 and 3).

### Normalization Equation
The equation for normalization is <br>
$$X' = \frac{X - X_{mean}}{\sqrt{X_{var}^{2} + \epsilon}}$$

Where <br>
$X'$: Normalized output<br>
$X$: Input<br>
$X_{mean}$: Mean of Input across it'ss dimensions<br>
$X_{var}$: Variance of Input across it's dimensions (biased in case for normalization for a batch but unbiased when updating running variance)<br>
$\epsilon$: Epsilon to prevent computational overflow<br>

**Note**:<br>
You see that there is mention of **biased** and **unbiased** variance mentioned above. This is because when we normalizing the batch we know the sample size and therefore we use biased variance as we know the size of the batch  which is N, but in casee of runnning variance we are inferring the populuation variance from the batch and hence to achieve the true population variance we need to be unbiased towards the population and therefore we need to use unbiased variance while updating the running variance.

## Modes of Batch Normalization
In batch normalizationn we have two modes in which the input gets normalized.

#### Mode 1: When in training
In training mode we want the input to get normalized based on the data part of the batch. We also want the current batch to contribute to the running mean and variance. The same mean can be used for normalization and adding value to the runnning mean but we need two different variances as mentioned above.

#### Mode 2: When in inference/test mode
In inference mode we want the input to get normalized but most of the times we wont have batched calls so for that we need running mean and running variance to be used to normalize the layer.

The outputs in both training and inference are scaled and shifted using $\lambda$ and $\beta$.

The complete implementation is done here `pytorch-fundamentals/src/pytorch_fundamentals/layers/normalization.py` for a Batch Normalization of a 4D tensor (batches of images)


## Conclusion
It is important to remember that batch normalization is done to speed up the process of training by making sure the distribution of activations in the hidden layers does not change allowing for faster information gain.